# Fooocus no Google Colab — um play e pronto

Gera **fotos** por inteligência artificial numa página com botões. É o painel de imagem
mais fácil que existe: você escreve o que quer e clica em Generate.

### Antes de apertar o play

Menu do topo → **Ambiente de execução** → **Alterar o tipo de ambiente de execução** →
marque **T4 GPU** → **Salvar**. Sem isso nada funciona.

### Depois

Aperte o play, autorize o Google Drive quando ele pedir, e **espere o link azul terminado
em `.gradio.live`**. Esse link é o painel.

**Não feche esta aba** enquanto estiver usando o painel.


In [ ]:
#@title ## Fooocus no Colab — aperte o play e espere o link azul { display-mode: "form" }
#@markdown Painel de **imagem** (fotos). Nao precisa mexer em nada aqui embaixo.
USAR_GOOGLE_DRIVE = True  #@param {type:"boolean"}
#@markdown Pasta dentro do seu Drive onde ficam as fotos geradas e as LoRAs:
PASTA_NO_DRIVE = "Fooocus"  #@param {type:"string"}

import os, json, subprocess

# ------------------------------------------------------------------ 1. a placa
gpu = ""
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except Exception:
    pass

if not gpu:
    print("\n" + "!" * 72)
    print("!!  PARE. ESTA SESSAO ESTA SEM PLACA DE VIDEO E O PAINEL NAO LIGA SEM ELA.")
    print("!!")
    print("!!    1) Menu do topo  ->  Ambiente de execucao")
    print("!!    2) Alterar o tipo de ambiente de execucao")
    print("!!    3) Marque  T4 GPU  ->  Salvar")
    print("!!    4) Aperte o play aqui de novo")
    print("!" * 72 + "\n")
    raise RuntimeError("Sem placa de video. Escolha T4 GPU em Ambiente de execucao e rode de novo.")

print("Placa de video: " + gpu)

# ------------------------------------------------------------------ 2. o Drive
PASTA = None
if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/' + PASTA_NO_DRIVE.strip('/ ')
    for sub in ('loras', 'saida'):
        os.makedirs(os.path.join(PASTA, sub), exist_ok=True)
    print("Drive conectado. Fotos e LoRAs em: " + PASTA)
else:
    print("Sem Drive: as fotos somem quando o Colab fechar. Baixe antes de sair.")

# ------------------------------------------------------------------ 3. o programa
!pip install -q pygit2==1.15.1
%cd /content
if not os.path.isdir('/content/Fooocus'):
    print("\nBaixando o programa Fooocus (uma vez so)...")
    !git clone -q https://github.com/lllyasviel/Fooocus.git
%cd /content/Fooocus

# ------------------------------------------------------------------ 4. mandar salvar no Drive
# Os modelos grandes ficam AQUI no Colab de proposito: baixar do site oficial e mais rapido
# do que ler 6 GB do Drive a cada vez que voce liga. So o que importa guardar vai para o Drive.
if PASTA:
    with open('config.txt', 'w', encoding='utf-8') as f:
        json.dump({
            "path_loras":   [os.path.join(PASTA, 'loras')],
            "path_outputs": os.path.join(PASTA, 'saida'),
        }, f, indent=2)
    print("Fooocus configurado para salvar as fotos no seu Drive.")

# ------------------------------------------------------------------ 5. ligar
print("\n" + "=" * 72)
print("LIGANDO O PAINEL. Na primeira vez ele baixa o modelo (uns 7 GB),")
print("entao o link pode levar de 5 a 10 minutos para aparecer. E normal.")
print("Quando aparecer, clique no endereco que termina em  .gradio.live")
print("=" * 72 + "\n")

!python entry_with_update.py --share --always-high-vram

